In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from src.dataLoader import getResultDataframe
from src.poissonModel import PoissonModel, poissonPdf
import polars as pl
import datetime
import numpy as np

In [3]:
df = getResultDataframe()
if df is None:
    df = pl.DataFrame()

In [4]:
df

date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
str,str,str,i64,i64,str,str,str,bool
"""1872-11-30""","""Scotland""","""England""",0,0,"""Friendly""","""Glasgow""","""Scotland""",false
"""1873-03-08""","""England""","""Scotland""",4,2,"""Friendly""","""London""","""England""",false
"""1874-03-07""","""Scotland""","""England""",2,1,"""Friendly""","""Glasgow""","""Scotland""",false
"""1875-03-06""","""England""","""Scotland""",2,2,"""Friendly""","""London""","""England""",false
"""1876-03-04""","""Scotland""","""England""",3,0,"""Friendly""","""Glasgow""","""Scotland""",false
…,…,…,…,…,…,…,…,…
"""2026-06-27""","""Jordan""","""Argentina""",null,null,"""FIFA World Cup""","""Arlington""","""United States""",true
"""2026-06-27""","""Colombia""","""Portugal""",null,null,"""FIFA World Cup""","""Miami Gardens""","""United States""",true
"""2026-06-27""","""DR Congo""","""Uzbekistan""",null,null,"""FIFA World Cup""","""Atlanta""","""United States""",true


In [5]:
allTeams = list(set(df["home_team"].unique().to_list() + df["away_team"].unique().to_list()))
model = PoissonModel(teamList=allTeams)

In [6]:
stats = model.getTeamParamsDf()
stats

team,attackingParams,defendingParams
str,f32,f32
"""Tibet""",0.972301,1.017163
"""Panjab""",1.044684,1.111448
"""Marshall Islands""",1.149584,1.05032
"""Mozambique""",1.002754,1.018319
"""Benin""",1.011858,0.969904
…,…,…
"""Gotland""",1.026222,0.901369
"""Menorca""",1.039879,0.976488
"""Australia""",1.027334,0.99696


In [7]:
model.getHomeMultiplier()

np.float32(0.95581186)

In [26]:
worldCupYear = 1998

In [27]:
scope = pl.lit(True)
scope = scope & (pl.col("date").is_between(datetime.date(worldCupYear-1, 6, 1), datetime.date(worldCupYear, 6, 1)))
scope = scope & (pl.all_horizontal([pl.col(column).is_not_null() for column in df.columns]))
model.fitPoisson(
    data=df,
    scopeExpr=scope,
    homeTeamExpr=pl.col("home_team"),
    awayTeamExpr=pl.col("away_team"),
    homeScoreExpr=pl.col("home_score"),
    awayScoreExpr=pl.col("away_score"),
    matchStatusExpr=pl.col("neutral"),
    numEpochs=10000
)

In [28]:
scopeTest = pl.lit(True)
scopeTest = scopeTest & (pl.col("tournament") == "FIFA World Cup")
scopeTest = scopeTest & (pl.col("date").is_between(datetime.date(worldCupYear, 1, 1), datetime.date(worldCupYear, 12, 31)))
dfTest = df.filter(scopeTest)

In [29]:
likelyhood = model.predictScore(
    dfTest,
    pl.col("home_team"),
    pl.col("away_team"),
    pl.col("home_score"),
    pl.col("away_score"),
    pl.when(~pl.col("neutral")).then(pl.col("home_team")).otherwise("neutral")
)

In [30]:
dfTest= dfTest.with_columns(likelyhood=likelyhood)

In [31]:
dfTest["likelyhood"].mean()

0.06951053169163102

In [ ]:
dfTest

date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,likelyhood
str,str,str,i64,i64,str,str,str,bool,f64
"""1998-06-10""","""Brazil""","""Scotland""",2,1,"""FIFA World Cup""","""Saint-Denis""","""France""",true,0.063426
"""1998-06-10""","""Morocco""","""Norway""",2,2,"""FIFA World Cup""","""Montpellier""","""France""",true,0.097104
"""1998-06-11""","""Cameroon""","""Austria""",1,1,"""FIFA World Cup""","""Toulouse""","""France""",true,0.110124
"""1998-06-11""","""Italy""","""Chile""",2,2,"""FIFA World Cup""","""Bordeaux""","""France""",true,0.054875
"""1998-06-12""","""France""","""South Africa""",3,0,"""FIFA World Cup""","""Marseille""","""France""",false,0.047203
…,…,…,…,…,…,…,…,…,…
"""1998-07-04""","""Netherlands""","""Argentina""",2,1,"""FIFA World Cup""","""Marseille""","""France""",true,0.094871
"""1998-07-07""","""Brazil""","""Netherlands""",1,1,"""FIFA World Cup""","""Marseille""","""France""",true,0.053212
"""1998-07-08""","""France""","""Croatia""",2,1,"""FIFA World Cup""","""Saint-Denis""","""France""",false,0.046065


In [15]:
model.getTeamParamsDf().filter(pl.col("team")=="France")

team,attackingParams,defendingParams
str,f32,f32
"""France""",3.800171,3.904303


In [16]:
model.getTeamParamsDf().filter(pl.col("team")=="Germany")

team,attackingParams,defendingParams
str,f32,f32
"""Germany""",4.339887,2.399774


In [18]:
dfTest

date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,likelyhood
str,str,str,i64,i64,str,str,str,bool,f64
"""2014-06-12""","""Brazil""","""Croatia""",3,1,"""FIFA World Cup""","""São Paulo""","""Brazil""",false,0.07603
"""2014-06-13""","""Chile""","""Australia""",3,1,"""FIFA World Cup""","""Cuiabá""","""Brazil""",true,0.036976
"""2014-06-13""","""Mexico""","""Cameroon""",1,0,"""FIFA World Cup""","""Natal""","""Brazil""",true,0.05124
"""2014-06-13""","""Spain""","""Netherlands""",1,5,"""FIFA World Cup""","""Salvador""","""Brazil""",true,0.115301
"""2014-06-14""","""Colombia""","""Greece""",3,0,"""FIFA World Cup""","""Belo Horizonte""","""Brazil""",true,0.084572
…,…,…,…,…,…,…,…,…,…
"""2014-07-05""","""Netherlands""","""Costa Rica""",0,0,"""FIFA World Cup""","""Salvador""","""Brazil""",true,0.017397
"""2014-07-08""","""Brazil""","""Germany""",1,7,"""FIFA World Cup""","""Belo Horizonte""","""Brazil""",false,0.063035
"""2014-07-09""","""Netherlands""","""Argentina""",0,0,"""FIFA World Cup""","""São Paulo""","""Brazil""",true,0.17744
